In [32]:
from pydoc import text

from debugpy.launcher import output
#LOAD ENV VARIABLES
from dotenv import load_dotenv

load_dotenv()

#Create an API client
import os
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
model = "gemini-3.1-flash-lite"
#noinspection PyTypeChecker
def add_user_message(messages,text):
    user_message = {"role": "user", "parts": [{"text": text}]}
    messages.append(user_message)

def add_assistant_message(messages,text):
    assistant_message = {"role": "model", "parts": [{"text": text}]}
    messages.append(assistant_message)

def chat(messages,system=None,stop_sequences=None):

    params = {
        "model":model,
        "contents":messages,
        "config":types.GenerateContentConfig(
            stop_sequences=stop_sequences
        )
    }
    if system:
        params["config"] = types.GenerateContentConfig(
            system_instruction=system
        )

    message = client.models.generate_content(**params)
    return message.text

In [33]:
import json

def generate_dataset():
    prompt=""""
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages,prompt)
    add_assistant_message(messages,"```json")
    text=chat(messages,stop_sequences=["```"])
    return json.loads(text)

In [34]:
dataset = generate_dataset()

with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [35]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [36]:
def run_prompt(test_case):
    """Merge the prompt and test case input, then return the result."""
    prompt = f"""
    Please solve the following task:
    {test_case["task"]}

    * Respond only with Python,JSON or plain Regex
    * Do not add and comment ,commentary or explanation
    """

    messages = []
    add_user_message(messages,prompt)
    add_assistant_message(messages, "```code")
    output= chat(messages)
    return output

In [37]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [38]:
def run_test_case(test_case):
    """Calls run_prompt, then grade the result"""
    output=run_prompt(test_case)

    model_grade=grade_by_model(test_case,output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score =grade_syntax(output,test_case)

    score = (model_score+syntax_score)/2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning":reasoning
    }

In [39]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [43]:
with open("dataset.json", "r") as f:
    dataset=json.load(f)

results = run_eval(dataset)

Average score: 5.666666666666667


In [44]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\n{\n    \"Version\": \"2012-10-17\",\n    \"Statement\": [\n        {\n            \"Effect\": \"Allow\",\n            \"Action\": \"s3:GetObject\",\n            \"Resource\": \"arn:aws:s3:::my-app-assets/*\"\n        }\n    ]\n}\n```",
    "test_case": {
      "task": "Generate an IAM policy JSON document that grants s3:GetObject permission to all objects within a specific bucket named 'my-app-assets'.",
      "format": "json",
      "solution_criteria": "The JSON must contain the correct 'Version', 'Statement' block, 'Effect': 'Allow', 'Action': ['s3:GetObject'], and a 'Resource' ARN string formatted as 'arn:aws:s3:::my-app-assets/*'."
    },
    "score": 5.0,
    "reasoning": "The generated JSON is syntactically correct, follows AWS IAM policy best practices for the requested scope, and adheres strictly to all provided criteria."
  },
  {
    "output": "\nimport boto3\n\ndef get_buckets_by_region(region_name):\n    s3 = boto3.client('s3')\n    s3_resource = bot